In [1]:
import os, multiprocessing as mp
print(mp.cpu_count(), os.cpu_count())


8 8


In [2]:
try:
 import cypari2
 pari = cypari2.Pari()
 print("cypari2 ok", cypari2.__version__ if hasattr(cypari2,'__version__') else '')
 print(pari.version())
except Exception as e:
 print("ERR", e)


ERR No module named 'cypari2'


In [3]:
%%bash
pip install cypari2 2>&1 | tail -5


_x86_64.whl.metadata (12 kB)

17_x86_64.manylinux_2_28_x86_64.whl (9.5 MB)

x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (268 kB)

nals, cypari2


In [4]:
import cypari2
pari = cypari2.Pari()
print(pari.version())
pari.default("realprecision", 50)
# Define L(Delta, s) using PARI: lfuninit for the modular form Delta (weight 12, level 1, trivial char)
# In PARI: L = lfunmf(mfinit([1,12],0), mffromell? ) — simpler: use lfunan or known constructor
# PARI provides: ldelta = lfunmf(mfinit([1,12]),mfDelta) — let's check
# Easier: use the L-function of an elliptic curve? No, Delta is the unique cusp form weight 12 level 1.
# PARI: mfDelta = mfinit([1,12],1); but simplest: use lfun with eta product? 
# Actually PARI has: lfunramanujan? No. We construct: mf = mfinit([1,12],1); D = mfeigenbasis(mf)[1]; L = lfunmf(mf,D)
mf = pari("mfinit([1,12],1)")
print(mf)
eig = pari("mfeigenbasis(mfinit([1,12],1))")
print(eig)


(2, 17, 2)
[[1, 12, [[[1, [0]], [1, [], []], [[]~, Vecsmall([])], [[], [], [;], [], [], []], [;]], []~, 1, t - 1], 1], [], [[[Vecsmall([9]), [1, 12, [[[1, [0]], [1, [], []], [[]~, Vecsmall([])], [[], [], [;], [], [], []], [;]], []~, 1, t - 1], y]], [[[1, [0]], [1, [], []], [[]~, Vecsmall([])], [[], [], [;], [], [], []], [;]], []~, 1, t - 1]]], [Vecsmall([1, 1, 1])], [Vecsmall([2]), [Mat(1), 1, 1, 0], [0; 1]], [0, 0, 0, 0, 0]]
[[[Vecsmall([15]), [1, 12, [[[1, [0]], [1, [], []], [[]~, Vecsmall([])], [[], [], [;], [], [], []], [;]], []~, 1, t - 1], y]], [[[Vecsmall([9]), [1, 12, [[[1, [0]], [1, [], []], [[]~, Vecsmall([])], [[], [], [;], [], [], []], [;]], []~, 1, t - 1], y]], [[[1, [0]], [1, [], []], [[]~, Vecsmall([])], [[], [], [;], [], [], []], [;]], []~, 1, t - 1]]], [1]~, 1]]


In [5]:
# Build L(Delta, s)
L = pari("L = lfunmf(mfinit([1,12],1))")
print(type(L))
# Check first few zeros
import time
t0 = time.time()
z = pari.lfunzeros(L, 30)
print("time", time.time()-t0)
print(z[:3])


<class 'cypari2.gen.Gen'>


PariError: incorrect type in lfunmisc_to_ldata (t_VEC)

In [6]:
# lfunmf returns a vector when there are multiple eigenforms; for level 1 weight 12, just one.
# We need the eigenform itself. Let's get it properly.
L = pari("lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
import time
t0 = time.time()
z = pari.lfunzeros(L, 30)
print("time", time.time()-t0)
print(z)


time 0.012488126754760742
[9.222379399921102523, 13.907549861392134407, 17.442776978234473314, 19.656513141954961000, 22.336103637209867274, 25.274636548112365356, 26.804391158350403034, 28.831682624186875444]


In [7]:
# Good. Now lfunzeros accepts [Tmin, Tmax] range or a single T (giving zeros up to height T).
# Test interval form:
import time
t0 = time.time()
z = pari.lfunzeros(L, [0, 30])
print("time", time.time()-t0)
print(z)


time 0.011168479919433594
[9.222379399921102523, 13.907549861392134407, 17.442776978234473314, 19.656513141954961000, 22.336103637209867274, 25.274636548112365356, 26.804391158350403034, 28.831682624186875444]


In [8]:
# Quick benchmarks at different heights to estimate scaling
import time
for T in [100, 200, 400, 800]:
 t0 = time.time()
 z = pari.lfunzeros(L, [0, T])
 dt = time.time()-t0
 print(f"T={T}: n_zeros={len(z)}, time={dt:.2f}s, last={z[-1]}")


IndexError: index out of range

In [9]:
import time
for T in [100, 200, 400, 800]:
 t0 = time.time()
 z = pari.lfunzeros(L, [0, T])
 dt = time.time()-t0
 n = len(z)
 last = z[n-1] if n>0 else None
 print(f"T={T}: n_zeros={n}, time={dt:.2f}s, last={last}")


T=100: n_zeros=59, time=0.11s, last=99.74664890030414031
T=200: n_zeros=159, time=0.50s, last=199.31123215186825786


T=400: n_zeros=404, time=3.75s, last=399.4108553650284787
T=800: n_zeros=982, time=40.14s, last=799.1140258006199608


In [10]:
# Confirms ~T^3 scaling. For T=4075 single-core would be huge.
# Test that interval [Tmin,Tmax] with Tmin>0 is faster than [0,Tmax]
import time
t0 = time.time()
z = pari.lfunzeros(L, [700, 800])
dt = time.time()-t0
print(f"[700,800]: n={len(z)}, time={dt:.2f}s")


[700,800]: n=152, time=10.71s


In [11]:
# Restricted intervals are indeed cheaper for high heights. Good — parallelization viable.
# Estimate density: 159 zeros at T=200; 982 at T=800. For N=5000 expect T~4075.
# Build worker script. Use multiprocessing 'spawn' to safely init separate PARI contexts.

worker_code = r'''
import os, sys, time, pickle
import cypari2

def worker(args):
 tmin, tmax, dps, idx = args
 pari = cypari2.Pari()
 pari.default("realprecision", dps)
 pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
 t0 = time.time()
 z = pari.lfunzeros(pari("L"), [tmin, tmax])
 elapsed = time.time() - t0
 n = len(z)
 # Convert to mpmath-friendly strings (preserve full precision)
 out = [str(z[i]) for i in range(n)]
 return idx, tmin, tmax, elapsed, out
'''
with open("/workspace/603406db-c27e-431d-80f1-941da3dcdfc0/delta_worker.py","w") as f:
 f.write(worker_code)
print("written")


written


In [12]:
# Design partition. Density rises with T. Roughly density of L(Delta) zeros ~ (1/pi)*log(T/(2*pi)) (analogue),
# so cumulative N(T) ~ T * log(T/(2 pi e))/(2 pi)? Let's empirically fit.
# T=200 -> 159; T=400 -> 404; T=800 -> 982. Use these to fit N(T) ~ a*T*log(T) + b*T form.
import numpy as np
from numpy.linalg import lstsq
Ts = np.array([100,200,400,800])
Ns = np.array([59,159,404,982])
# model N = c1 * T * log(T) + c2 * T
A = np.column_stack([Ts*np.log(Ts), Ts])
c,_,_,_ = lstsq(A, Ns, rcond=None)
print(c)
# Predict T for N=5000
from scipy.optimize import brentq
f = lambda T: c[0]*T*np.log(T) + c[1]*T - 5000
T_pred = brentq(f, 1000, 10000)
print("T predicted for 5000:", T_pred)


[ 0.31124732 -0.85333237]


T predicted for 5000: 3043.0898693723216


In [13]:
# The spec says T~4075 for N=5000. We use T_max=4075 with a small safety margin and then trim to first 5000.
# Partition by *equal-cost* (cost ~ length * log(T)^? actually proportional to # of zeros in interval, but each evaluation expensive).
# Use equal width slices first then trim. Use K=8 workers.
# Run dry-run on small partition first to verify correctness.

import sys, time
sys.path.insert(0, "/workspace/603406db-c27e-431d-80f1-941da3dcdfc0")

# Need spawn context because PARI/cypari2 is initialized in main process already
import multiprocessing as mp
ctx = mp.get_context("spawn")

def run_partition(intervals, dps, processes):
 from delta_worker import worker
 t0 = time.time()
 with ctx.Pool(processes=processes) as pool:
 results = pool.map(worker, [(a,b,dps,i) for i,(a,b) in enumerate(intervals)])
 wall = time.time() - t0
 return wall, results

# Small test: serial vs parallel on [0,400]
intervals_small = [(0,200),(200,400)]
wall, res = run_partition(intervals_small, 50, 2)
print(f"parallel 2 workers [0-200,200-400]: wall={wall:.2f}s")
for r in res:
 print(f" idx={r[0]} [{r[1]},{r[2]}] elapsed={r[3]:.2f}s n={len(r[4])}")


parallel 2 workers [0-200,200-400]: wall=3.17s
 idx=0 [0,200] elapsed=0.53s n=159
 idx=1 [200,400] elapsed=2.79s n=245


In [14]:
# Good — parallel works. Now design partition for [0, 4075].
# Cost per worker is dominated by the highest height in its interval (~T^3 like).
# To balance: assign workers based on equalizing integral of T^2 dT (proxy for cumulative cost).
# Actually we observed runtime to lfunzeros([Tmin,Tmax]) at T=700-800 was 10.7s. Single evaluation cost ~ T^? 
# But lfunzeros([Tmin,Tmax]) cost is roughly (#zeros in interval) * (eval cost at T~Tmax).
# Eval cost grows with T. Use cost_density(T) = T * log(T) approximately for cost per unit T at height T? 
# Simpler: pick partition that gives roughly equal expected times via cumulative cost ~ T^3.

import numpy as np
T_max = 4075
K = 8
# Equal "T^3" partition: x_i such that x_i^3 / T_max^3 = i/K
boundaries = [T_max * (i/K)**(1/3) for i in range(K+1)]
print("T^3-equal boundaries:", [f"{b:.1f}" for b in boundaries])


T^3-equal boundaries: ['0.0', '2037.5', '2567.1', '2938.6', '3234.3', '3484.1', '3702.4', '3897.6', '4075.0']


In [15]:
# That puts most workers in [2000, 4075] which is right because cost grows steeply.
# But interval [0, 2037] still has many zeros and substantial cost. Let me instead think about
# cost = integral_a^b (density(T) * eval_cost(T)) dT ~ integral T*log(T) * T dT ~ integral T^2 log T.
# Using T^3 partition is close. Let's use it.

intervals = [(boundaries[i], boundaries[i+1]) for i in range(K)]
# Make sure first interval starts at 0 (already does)
print(intervals)


[(0.0, 2037.5), (2037.5, 2567.0891391608043), (2567.0891391608043, 2938.5834995013442), (2938.5834995013442, 3234.3296433852065), (3234.3296433852065, 3484.0759913537704), (3484.0759913537704, 3702.3832078954847), (3702.3832078954847, 3897.597284898743), (3897.597284898743, 4075.0)]


In [16]:
# Before launching the big job, let me check that PARI's lfunzeros on [Tmin, Tmax] with Tmin>0
# does NOT miss zeros at boundaries — verify uniqueness when concatenating
# Test: zeros in [0,400] vs [0,200] + [200,400]
import time
t0=time.time()
za = pari.lfunzeros(L, [0,400])
print(f"[0,400]: {len(za)} time {time.time()-t0:.2f}")
t0=time.time()
zb1 = pari.lfunzeros(L, [0,200])
zb2 = pari.lfunzeros(L, [200,400])
print(f"[0,200]+[200,400]: {len(zb1)+len(zb2)} time {time.time()-t0:.2f}")
# Compare last of zb1 and first of zb2
print("last [0,200]:", zb1[len(zb1)-1])
print("first [200,400]:", zb2[0])


[0,400]: 404 time 3.67
[0,200]+[200,400]: 404 time 3.32
last [0,200]: 199.31123215186825786
first [200,400]: 200.14614120290689498


In [17]:
# Perfect — counts match, no overlap. Now run the full job.
# Budget concern: predicted T for N=5000 was ~3043 by extrapolation, but spec says 4075.
# We use 4075. Let me also estimate runtime: serial [0,800] took ~40s. Cost scales ~T^3 so 
# [0,4075] would be ~40 * (4075/800)^3 = ~5300s = 88min single core. With 8-way balanced partition: ~700s expected.
# Let's launch.

import time
t0 = time.time()
wall, results = run_partition(intervals, 50, 8)
print(f"\nTotal wall time: {wall:.1f}s")
total_zeros = 0
for r in sorted(results, key=lambda x: x[0]):
 print(f" idx={r[0]} [{r[1]:.1f},{r[2]:.1f}] elapsed={r[3]:.1f}s n={len(r[4])}")
 total_zeros += len(r[4])
print(f"Total zeros found: {total_zeros}")


TimeoutError: Code execution timed out after 900.0 seconds

In [18]:
# Cell timed out at 900s. The pool workers may still be running. Let's run the whole thing via a script in background.
# Move execution to a standalone script that writes results to disk; poll for completion.

script = r'''
import os, sys, time, pickle, json
import multiprocessing as mp

sys.path.insert(0, "/workspace/603406db-c27e-431d-80f1-941da3dcdfc0")
from delta_worker import worker

T_max = 4075.0
K = 8
boundaries = [T_max * (i/K)**(1/3) for i in range(K+1)]
intervals = [(boundaries[i], boundaries[i+1]) for i in range(K)]
dps = 50

if __name__ == "__main__":
 ctx = mp.get_context("spawn")
 t0 = time.time()
 log = open("/workspace/603406db-c27e-431d-80f1-941da3dcdfc0/delta_run.log","w")
 log.write(f"Starting K={K} workers, intervals={intervals}\n"); log.flush()
 with ctx.Pool(processes=K) as pool:
 results = []
 for r in pool.imap_unordered(worker, [(a,b,dps,i) for i,(a,b) in enumerate(intervals)]):
 elapsed = time.time()-t0
 log.write(f"[wall {elapsed:.1f}s] worker idx={r[0]} [{r[1]:.1f},{r[2]:.1f}] elapsed={r[3]:.1f}s n_zeros={len(r[4])}\n"); log.flush()
 results.append(r)
 wall = time.time() - t0
 log.write(f"Total wall: {wall:.1f}s\n"); log.flush()
 with open("/workspace/603406db-c27e-431d-80f1-941da3dcdfc0/delta_zeros_raw.pkl","wb") as f:
 pickle.dump({"wall": wall, "intervals": intervals, "results": results, "dps": dps}, f)
 log.write("Saved delta_zeros_raw.pkl\n")
 log.close()
'''
with open("/workspace/603406db-c27e-431d-80f1-941da3dcdfc0/run_delta.py","w") as f:
 f.write(script)

# Launch as a background process
import subprocess, os
# Check if already running
proc = subprocess.Popen(
 ["python", "/workspace/603406db-c27e-431d-80f1-941da3dcdfc0/run_delta.py"],
 stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
 start_new_session=True,
)
print("PID:", proc.pid)


TimeoutError: Code execution timed out after 900.0 seconds

In [19]:
# Likely the kernel still has workers in pool from prior interrupted call. Let me reset kernel.
print("reset needed")


TimeoutError: Code execution timed out after 900.0 seconds

In [ ]:
# KERNEL_RESET

In [1]:
import subprocess, os, time
# Check if any python processes are running our script
r = subprocess.run(["pgrep","-af","run_delta.py"], capture_output=True, text=True)
print("RUN_DELTA:", r.stdout)
r = subprocess.run(["ls","-la","/workspace/603406db-c27e-431d-80f1-941da3dcdfc0/"], capture_output=True, text=True)
print(r.stdout)


PermissionError: [Errno 13] Permission denied: 'pgrep'

In [2]:
import subprocess, os, time
r = subprocess.run(["ls","-la","/workspace/603406db-c27e-431d-80f1-941da3dcdfc0/"], capture_output=True, text=True)
print(r.stdout)
# Check log if exists
p = "/workspace/603406db-c27e-431d-80f1-941da3dcdfc0/delta_run.log"
if os.path.exists(p):
 print("---LOG---")
 print(open(p).read())


total 50
drwxrwsr-x 6 root jupytergroup 4096 May 30 19:24 .
drwxr-xr-x 5 root root 4096 May 30 19:22 ..
drwxrwsr-x 5 root jupytergroup 4096 May 30 19:22 .config
-rw-rw-r-- 1 kerneluser jupytergroup 0 May 30 19:22 .kernel_llm_logs_1.txt
drwxrwsr-x 2 root jupytergroup 4096 May 30 19:22 .prompts
-rw-r--r-- 1 root jupytergroup 25848 May 30 19:22 -PROMPT-v6-DATASET.md
drwxrwsr-x 2 kerneluser jupytergroup 4096 May 30 19:24 __pycache__
-rw-rw-r-- 1 kerneluser jupytergroup 500 May 30 19:24 delta_worker.py
drwxr-sr-x 2 root jupytergroup 4096 May 30 19:22 memory



In [3]:
# Background script was never launched (the launch cell timed out before reaching subprocess.Popen).
# Time budget left: ~10 minutes. A full N=5000 run is infeasible now (estimated several thousand seconds parallel).
# Strategy: produce a smaller validated run (e.g., N~500-800 zeros up to T~700) to demonstrate parallel speedup
# vs serial baseline, validate γ₁ to >=30 decimal places, save a partial cache, and document infeasibility honestly.

# Re-write worker module since kernel was reset
worker_code = r'''
import os, sys, time, pickle
import cypari2

def worker(args):
 tmin, tmax, dps, idx = args
 pari = cypari2.Pari()
 pari.default("realprecision", dps)
 pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
 t0 = time.time()
 z = pari.lfunzeros(pari("L"), [tmin, tmax])
 elapsed = time.time() - t0
 n = len(z)
 out = [str(z[i]) for i in range(n)]
 return idx, tmin, tmax, elapsed, out
'''
with open("/workspace/603406db-c27e-431d-80f1-941da3dcdfc0/delta_worker.py","w") as f:
 f.write(worker_code)
print("ok")


ok


In [4]:
# Establish a serial baseline and parallel run on a tractable range [0, T_demo].
# Choose T_demo to fit within ~3-4 minutes serially. From earlier: [0,800] = 40s serial. 
# Try T_demo=1200. Predicted serial ~ 40*(1200/800)^3 = 135s. Parallel with 8 should be <30s.

import sys, time, multiprocessing as mp
sys.path.insert(0, "/workspace/603406db-c27e-431d-80f1-941da3dcdfc0")
from delta_worker import worker

T_demo = 1200.0
K = 8
boundaries = [T_demo * (i/K)**(1/3) for i in range(K+1)]
intervals = [(boundaries[i], boundaries[i+1]) for i in range(K)]
print("intervals:", [(round(a,1),round(b,1)) for a,b in intervals])

ctx = mp.get_context("spawn")
t0 = time.time()
with ctx.Pool(processes=K) as pool:
 results = pool.map(worker, [(a,b,50,i) for i,(a,b) in enumerate(intervals)])
wall_par = time.time() - t0
print(f"\nParallel wall: {wall_par:.1f}s")
for r in sorted(results, key=lambda x: x[0]):
 print(f" idx={r[0]} [{r[1]:.1f},{r[2]:.1f}] elapsed={r[3]:.2f}s n={len(r[4])}")


intervals: [(0.0, 600.0), (600.0, 756.0), (756.0, 865.3), (865.3, 952.4), (952.4, 1026.0), (1026.0, 1090.3), (1090.3, 1147.8), (1147.8, 1200.0)]



Parallel wall: 30.2s
 idx=0 [0.0,600.0] elapsed=26.35s n=683
 idx=1 [600.0,756.0] elapsed=26.95s n=231
 idx=2 [756.0,865.3] elapsed=26.15s n=170
 idx=3 [865.3,952.4] elapsed=26.28s n=138
 idx=4 [952.4,1026.0] elapsed=27.34s n=118
 idx=5 [1026.0,1090.3] elapsed=28.12s n=105
 idx=6 [1090.3,1147.8] elapsed=28.51s n=95
 idx=7 [1147.8,1200.0] elapsed=29.63s n=87
